# Advance Retrievers in RAG

- Vector Store-backed Retriever
- Multi-Query Retriever
- Self-Querying Retriever
- Parent Document Retriever



In [ ]:
from langchain_openai import ChatOpenAI
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.retrievers.multi_query import MultiQueryRetriever

from langchain_core.documents import Document
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from lark import lark

from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain.storage import InMemoryStore

# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [ ]:
def llm():
    model_name = "gpt-4.1-nano"  # or "gpt-3.5-turbo"
    openai_llm = ChatOpenAI(
        model=model_name,
        temperature=0.5,
        max_tokens=256,
    )
    return openai_llm

In [4]:
def text_splitter(data, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

In [6]:
def openai_embedding():
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
    return embeddings

## Use Retrievers

## Vector Store-Backed Retriever

- Its a type of retriever that utilizes a vector store to fetch documents.
- This retriever leverages the search methods implemented by the vector store, such as `similarity search` and `Maximum Marginal Relevance (MMR)`, to query texts stored within it.

In [ ]:
### Download Company Policies document
# !wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"

In [ ]:
loader = TextLoader("companyPolicies.txt")
txt_data = loader.load() # txt_data only contain 1 document of complete text file

In [21]:
print(txt_data[0].metadata)
print(txt_data[0].page_content[:100])

{'source': 'companyPolicies.txt'}
1.	Code of Conduct

Our Code of Conduct outlines the fundamental principles and ethical standards th


In [23]:
chunks_txt = text_splitter(txt_data, 200, 20) # chunk_size is 200 and chunk_overlap is 20

In [ ]:
vectordb = Chroma.from_documents(chunks_txt, openai_embedding())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Simple similarity search

simple similarity search based on the vector database.

In [28]:
query = "email policy"
retriever = vectordb.as_retriever(search_kwargs={"k": 2}) # default retrieved documents are 4

docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and')]

### MMR search

MMR in vector stores is a technique used to balance the relevance and diversity of retrieved results.\
It selects documents that are both highly relevant to the query and minimally similar to previously selected documents.\
This approach helps to avoid redundancy and ensures a more comprehensive coverage of different aspects of the query.

In [29]:
retriever = vectordb.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='form of unlawful bias. This policy applies to every individual within the organization, including employees, contractors, visitors, and clients.'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='company values and contribute to our continued success. We continuously review and update this policy to reflect evolving best practices in recruitment.')]

### Similarity score threshold retrieval

Return only those documents which are above specified threshold

In [33]:
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.2}
)
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and')]

## Multi-Query Retriever

The `MultiQueryRetriever` is designed to overcome "distance-based" search limitations.

It uses an LLM to generate multiple queries from different perspectives for a given user input query.\
For each query, it retrieves a set of relevant documents and then takes the unique union of these results to form a larger set of potentially relevant documents.

In [35]:
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()

In [45]:
print(pdf_data[2].metadata)
print(pdf_data[2].page_content[:200])

{'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'page': 2}
 
Figure 2. An AIMessage illustration  
C. Prompt Template  
Prompt templates  [10] allow you to structure  input for LLMs. 
They provide a convenient way to format user inputs and 
provide instructio


In [ ]:
chunks_pdf = text_splitter(pdf_data, 500, 20)

# Delete existing VectorDB data
ids = vectordb.get()["ids"]
vectordb.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.

vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=openai_embedding())

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), 
    llm=llm(),
    ### We can also create custom prompt to generate different query versions
    # prompt=CUSTOM_PROMPT,
    # parser_key='lines'
)

In [50]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [51]:
docs = retriever.invoke(query)
docs

INFO:langchain.retrievers.multi_query:Generated queries: ["1. Can you summarize the main points the paper makes regarding LangChain's features and applications?  ", '2. How does the paper evaluate the effectiveness or limitations of LangChain in its proposed use cases?  ', '3. What insights does the paper provide about the role of LangChain in the broader context of language model frameworks?']


[Document(metadata={'page': 1, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you . Its'),
 Document(metadata={'page': 0, 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf'}, page_content='II. LANGCHAIN  \nLangChain , with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered

## Self-Querying Retriever

A `Self-Querying Retriever` has the ability to query itself. 

It not only use the user-input query for semantic similarity comparison with the contents of stored documents\
but also to extract and apply filters based on the metadata of those documents.

In [57]:
# Define a docs list
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

In [58]:
# Provide brief information about metadata fields. It will help llm to query based on metadata.
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

In [63]:
# Delete existing VectorDB data
ids = vectordb.get()["ids"]
vectordb.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.

vectordb = Chroma.from_documents(docs, openai_embedding())

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionDeleteEvent: capture() takes 1 positional argument but 3 were given


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [64]:
document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

In [65]:
# This example only specifies a filter
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9, 'year': 1979}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

In [66]:
# This example specifies a query and a filter
retriever.invoke("Has Greta Gerwig directed any movies about women")

[Document(metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(metadata={'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9, 'year': 1979}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'genre': 'animated', 'year': 1995}, page_content='Toys come alive and have a blast doing so'),
 Document(metadata={'genre': 'science fiction', 'rating': 7.7, 'year': 1993}, page_content='A bunch of scientists bring back dinosaurs and mayhem breaks loose')]

## Parent Document Retriever

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data.\
During retrieval, it first fetches the small chunks but then looks up the parent IDs for those chunks and returns those larger documents.

In [70]:
# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [71]:
vectordb = Chroma(
    collection_name="split_parents", embedding_function=openai_embedding()
)
#vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=watsonx_embedding())

# The storage layer for the parent documents
store = InMemoryStore() 

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [73]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [74]:
retriever.add_documents(txt_data) # Adding documents here instead of while creating Chroma instance

In [ ]:
len(list(store.yield_keys())) # Return number of large chunks.

19

In [84]:
sub_docs = vectordb.similarity_search("smoking policy") # retrieve small chunk
sub_docs

[Document(metadata={'doc_id': '9d22922c-38ab-46f2-9881-4e2da98b4dea', 'source': 'companyPolicies.txt'}, page_content='5.\tSmoking Policy'),
 Document(metadata={'doc_id': '0d4aa6fc-8b53-4773-a486-746e0a4a46a3', 'source': 'companyPolicies.txt'}, page_content='5.\tSmoking Policy'),
 Document(metadata={'doc_id': '9d22922c-38ab-46f2-9881-4e2da98b4dea', 'source': 'companyPolicies.txt'}, page_content='Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.'),
 Document(metadata={'doc_id': '9afbc276-8ba5-45a3-946f-13059436d603', 'source': 'companyPolicies.txt'}, page_content='We appreciate your cooperation in maintaining a smoke-free and safe environment for all.\n6.\tDrug and Alcohol Policy')]

In [85]:
retrieved_docs = retriever.invoke("smoking policy") # retrieve the relevant large chunk.
retrieved_docs

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='5.\tSmoking Policy\nPolicy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.\nDesignated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.\nSmoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.\nCompliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.'),
 Document(metadata={'source': 'companyPolicies.txt'}, page_content='Cost Manageme